# 로그인과 비밀번호

In [1]:
import os          
import pathlib     
                   
import sys         

here = pathlib.Path.cwd()      

ROOT = here.parents[2] if here.name == "day01" else here
os.chdir(ROOT)                 

print ("프로젝트 루트 : ", ROOT)

BACKEND = ROOT / "backend"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

프로젝트 루트 :  /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice


* bcrypt 암호화

In [2]:
import bcrypt

# hashed = bcrypt.hashpw("비밀번호",encode("utf-8"), bcrypt.gensalt()) # 암호화
# matched = bcrypt.checkpw("비밀번호".encode("utf-8"), hashed) # True /False
RAW = "hanwha2026!" # 비밀번호

first = bcrypt.hashpw(RAW.encode("utf-8"), bcrypt.gensalt()).decode("utf-8")
second = bcrypt.hashpw(RAW.encode("utf-8"), bcrypt.gensalt()).decode("utf-8")

print("first : ", first)
print("second : ", second)
print("해시 길이 : ", len(first), "자")

print("해시 검증 : ", bcrypt.checkpw(RAW.encode("utf-8"), first.encode("utf-8")))
print("해시 검증 : ", bcrypt.checkpw(RAW.encode("utf-8"), second.encode("utf-8")))
print("틀린비번 검증 : ", bcrypt.checkpw("한화2026!".encode("utf-8"), first.encode("utf-8")))

first :  $2b$12$IIKalFGHDuf5Ymx6ttcjBOXgU7WYmYYu9xtTdjfhUlQxkTA78XQDW
second :  $2b$12$haqwH06ynHUeUpMGeVOThOGGkFE2LyAFvB2M.i8DTTsAzxQlo43dW
해시 길이 :  60 자
해시 검증 :  True
해시 검증 :  True
틀린비번 검증 :  False


In [2]:
from sqlalchemy import inspect

from app.db.init_db import init_db
from app.db.session import get_engine

DB_FILE = ROOT / "app.db"        


get_engine().dispose()      # 풀에 남아있는 커넥션 먼저 닫기     
DB_FILE.unlink(missing_ok=True)  # DB파일 링크 끊기
init_db()   # 데이터베이스 초기화

columns = [c["name"] for c in inspect(get_engine()).get_columns("users")]
print("users 컬럼 :", columns)
print("password_hash 가 있는가 :", "password_hash" in columns)

users 컬럼 : ['id', 'emp_no', 'name', 'dept_id', 'role', 'clearance', 'password_hash', 'created_at', 'updated_at']
password_hash 가 있는가 : True


* 시드 결과 확인

In [3]:
from sqlalchemy import select
from sqlalchemy.orm import Session

from app.db.seed import seed_all
from app.db.seed_data import TEMP_PASSWORD
from app.models.org import User

print("시드 결과 :", seed_all())
print()

with Session(get_engine()) as s:
    for emp_no in ("2016-0231", "2011-0003"):
        user = s.scalars(select(User).where(User.emp_no == emp_no)).one()
        print(emp_no, user.name, " password_hash :",
              user.password_hash[:4] + "...", len(user.password_hash), "자")
    stored_hashes = s.scalars(select(User.password_hash)).all()

print()
print("평문이 그대로 저장된 사람이 있는가 :", TEMP_PASSWORD in stored_hashes)
print("일곱 명이 같은 해시를 쓰는가       :", len(set(stored_hashes)) == 1)


시드 결과 : {'departments': 6, 'users': 7, 'documents': 7, 'versions': 8}

2016-0231 김지원  password_hash : $2b$... 60 자
2011-0003 한서영  password_hash : $2b$... 60 자

평문이 그대로 저장된 사람이 있는가 : False
일곱 명이 같은 해시를 쓰는가       : True


In [8]:
from app.services import auth_service

print("로그인 성공 :", auth_service.authenticate("2016-0231", TEMP_PASSWORD))
print()

# 두 실패가 정말 구별되지 않는지 나란히 놓고 본다.
for emp_no, password, label in [
    ("2016-0231", "틀린비밀번호", "비밀번호가 틀렸을 때"),
    ("0000-0000", TEMP_PASSWORD, "사번이 아예 없을 때"),
]:
    try:
        auth_service.authenticate(emp_no, password)
    except AuthFailed as exc:
        print(label, ":", exc.status_code, exc.code, exc.message)


AuthFailed: 사번 또는 비밀번호가 올바르지 않습니다.